In [ ]:
import pandas as pd

# Read the first CSV file into a DataFrame
repairs = pd.read_csv("../../datasets/none_of_repairs.csv")

repairs

In [ ]:
len(repairs[(repairs['Constraint Deleted'] == True)])

In [ ]:
len(repairs[(repairs['Constraint Deprecated'] == True)])

In [ ]:
len(repairs[(repairs['Included as Exception'] == True)])

In [ ]:
len(repairs[(repairs['Constraint Deleted'] == True) &
               repairs['Included as Exception'] == True
           ])

In [ ]:
len(repairs[(repairs['T-box: Remove the value from the list of prohibited values'] == True) &
           repairs['Constraint Deleted'] == True])

In [ ]:
len(repairs[(repairs['T-box: Remove the value from the list of prohibited values'] == True) &
           repairs['Constraint Deleted'] == False])

In [ ]:
repairs[(repairs['T-box: Remove the value from the list of prohibited values'] == True) & (repairs['Constraint Deleted'] == False)]

In [ ]:
import requests
import xml.etree.ElementTree as ET

def existsTriple(subject, property, obj):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    property = property.replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/direct/")
    # SPARQL query
    query = f"ASK {{ <{obj}> <{property}> <{subject}> }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)

    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
subject = "http://www.wikidata.org/entity/Q10378015"
property = "http://www.wikidata.org/prop/direct/P1435"
obj = "http://www.wikidata.org/entity/Q2065736"
print(existsTriple(subject, property, obj))

In [ ]:
repairs['statement_deleted'] = False

In [ ]:
repairs

In [ ]:
for index, row in repairs.iterrows():
    # Extract subject and property without the prefix "http://www.wikidata.org/entity/"
    subject = row['Subject']
    property = row['Property']
    obj = row['Object']
    
    if (index % 200 == 0):
        print(index)
    #break
    
    if (row['Constraint Deleted'] == False and row['Constraint Deprecated'] == False and
        row['Included as Exception'] == False and row['T-box: Remove the value from the list of prohibited values'] == False
       ):
        # Call symmetricAdded function
        symm = not existsTriple(subject, property, obj)

        # Update instanceRemoved column
        repairs.at[index, 'statement_deleted'] = symm

# Display the updated DataFrame
#print(filtered_df)

In [ ]:
len(repairs[(repairs['statement_deleted'] == True)])

In [ ]:
repairs.to_csv('none_of_repairs_final.csv', index=False)